# Stage 7: Feature selection and preprocessing
1. **Exclude protected attributes** from model inputs (responsible-lending decision, documented in config)
2. **Split** into train / validation / test *before* anything is learned from the data
3. **Select features** with an unsupervised filter fitted on the training split only
4. **Handle class imbalance** with class weights
5. **Build reusable sklearn pipelines** (`ColumnTransformer`), one for linear models and one for trees

| Split | Share | Used for |
|---|---|---|
| Train | 70% | Fitting preprocessing + models |
| Validation | 15% | Early stopping, probability calibration, threshold choice, model selection |
| Test | 15% | **Touched once**, for the final unbiased report |

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, roc_auc_score

from credit_risk.artifacts import load_features, write_json
from credit_risk.config import get_paths, load_config
from credit_risk.preprocessing import (
    class_balance,
    get_feature_lists,
    make_linear_preprocessor,
    make_tree_preprocessor,
    prepare_X,
    save_feature_lists,
    save_split_ids,
    select_features,
    split_data,
)

cfg = load_config()
paths = get_paths(cfg)
ID, TARGET = cfg["data"]["id_column"], cfg["data"]["target"]
SEED = cfg["project"]["random_state"]

features = load_features(paths, "train")
print(features.shape)

(307511, 244)


## 1. Responsible-lending exclusions
`CODE_GENDER` and `NAME_FAMILY_STATUS` are prohibited bases for credit decisions in many jurisdictions (e.g. sex and
marital status under the US Equal Credit Opportunity Act). They are **kept in the data for fairness testing** but are
**not model inputs**. `AGE_GROUP` is only a reporting band.

In [2]:
exclude = cfg["fairness"]["exclude_from_model"]
num_cols, cat_cols = get_feature_lists(features, ID, TARGET, exclude=exclude)
print(f"Excluded from model: {exclude}")
print(f"Candidate features: {len(num_cols)} numeric + {len(cat_cols)} categorical")

Excluded from model: ['CODE_GENDER', 'NAME_FAMILY_STATUS', 'AGE_GROUP']
Candidate features: 227 numeric + 12 categorical


## 2. Train / validation / test split (stratified)
Home Credit has no application date, so an out-of-time split isn't possible. Stratifying keeps the default rate
identical across splits. In a real bank we would hold out the most recent origination months instead.

In [3]:
train, valid, test = split_data(features, TARGET, cfg["split"]["test_size"], cfg["split"]["valid_size"], SEED)
save_split_ids(train, valid, test, paths["processed_data"] / "split_ids.parquet", ID)
pd.DataFrame({s: class_balance(d[TARGET]) for s, d in [("train", train), ("valid", valid), ("test", test)]}).T

,n,defaults,default_rate,imbalance_ratio
train,215257.0,17377.0,0.080727,11.387466
valid,46127.0,3724.0,0.080734,11.386412
test,46127.0,3724.0,0.080734,11.386412


## 3. Feature selection (fitted on TRAIN only, target-free)
- drop features that are more than 95% missing
- drop near-constant features (one value covers more than 99.5% of rows)
- drop one of each near-duplicate pair (|Spearman ρ| > 0.98)

We deliberately avoid target-based selection here. The boosted models regularise themselves, and SHAP (Stage 11)
shows what each feature contributes. Target-based selection on all the data is a subtle form of leakage.

In [4]:
fs = cfg["feature_selection"]
num_sel, cat_sel, dropped = select_features(train, num_cols, cat_cols, fs["max_missing_rate"], fs["max_dominant_share"], fs["max_correlation"])
print(f"Kept {len(num_sel)} numeric + {len(cat_sel)} categorical features; dropped {len(dropped)}")
display(dropped["reason"].str.split(" ").str[0].value_counts().to_frame("dropped"))
dropped.head(15)

Kept 188 numeric + 12 categorical features; dropped 39


,dropped
reason,
corr>0.98,23
near_constant,16


,feature,reason,value
0,FLAG_MOBIL,near_constant,1.0000
1,FLAG_CONT_MOBILE,near_constant,0.9981
2,FLAG_DOCUMENT_2,near_constant,1.0000
3,FLAG_DOCUMENT_4,near_constant,0.9999
4,FLAG_DOCUMENT_7,near_constant,0.9998
5,FLAG_DOCUMENT_9,near_constant,0.9961
6,FLAG_DOCUMENT_10,near_constant,1.0000
7,FLAG_DOCUMENT_11,near_constant,0.9962
8,FLAG_DOCUMENT_12,near_constant,1.0000
9,FLAG_DOCUMENT_13,near_constant,0.9966


In [5]:
save_feature_lists(num_sel, cat_sel, paths["models"] / "feature_lists.json")
dropped.to_csv(paths["reports"] / "feature_selection_log.csv", index=False)

## 4. Preprocessing pipelines
| Step | Linear model (Logistic Regression) | Tree models (XGBoost, LightGBM) |
|---|---|---|
| Missing numeric | Median imputation **+ missing-indicator columns** | Left as `NaN` (trees learn a default direction) |
| Outliers | Winsorise at 0.5% / 99.5% quantiles learned on train | Not needed (split-based) |
| Scaling | StandardScaler | Not needed |
| Categorical | One-hot; rare levels (<0.5%) grouped as "infrequent" | Ordinal codes; unseen levels → -1 |

Every step is fitted on **train only** inside a `ColumnTransformer`, then saved together with the model, so the
app applies the identical transformation.

In [6]:
X_train = prepare_X(train, num_sel, cat_sel)
X_valid = prepare_X(valid, num_sel, cat_sel)

linear_prep = make_linear_preprocessor(num_sel, cat_sel)
tree_prep = make_tree_preprocessor(num_sel, cat_sel)
Xl = linear_prep.fit_transform(X_train)
Xt = tree_prep.fit_transform(X_train)
print(f"Linear design matrix: {Xl.shape}  (incl. {sum(c.startswith('missingindicator') for c in Xl.columns)} missing indicators)")
print(f"Tree design matrix:   {Xt.shape}")
linear_prep

Linear design matrix: (215257, 432)  (incl. 149 missing indicators)
Tree design matrix:   (215257, 200)


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The

In [7]:
# Proof that the scaler was fitted on train only: train means are ~0, validation means are not exactly 0
Xl_valid = linear_prep.transform(X_valid)
scaled_num = [c for c in num_sel if c in Xl.columns][:5]
pd.DataFrame({"train_mean": Xl[scaled_num].mean(), "valid_mean": Xl_valid[scaled_num].mean()}).round(3)

,train_mean,valid_mean
FLAG_OWN_CAR,0.0,-0.005
FLAG_OWN_REALTY,0.0,-0.003
CNT_CHILDREN,-0.0,0.012
AMT_INCOME_TOTAL,0.0,-0.000
AMT_CREDIT,0.0,-0.004


## 5. Class imbalance
Options and why we chose **class weights**:
| Approach | Effect | Verdict |
|---|---|---|
| Do nothing | Model optimises accuracy-like loss; predicted PDs stay low | Fine for ranking (AUC), poor recall at 0.5 |
| Random undersampling | Throws away ~90% of good borrowers | Wasteful at this size |
| SMOTE (synthetic defaulters) | Invents borrowers by interpolating; distorts probabilities | Rarely helps on large tabular credit data |
| **Class weights / `scale_pos_weight`** | Each defaulter counts ~11× in the loss | ✓ Simple, no data invented |

Any re-weighting **inflates** the predicted probabilities, so we **re-calibrate** on the validation set (Stage 8) to get
true PDs back. A PD feeds expected-loss and pricing calculations, so it has to mean what it says.

In [8]:
bal = class_balance(train[TARGET])
pos_weight = bal["imbalance_ratio"]
print(f"scale_pos_weight = {pos_weight:.2f}")

y_tr, y_va = train[TARGET], valid[TARGET]
demo = {}
for label, cw in [("no weighting", None), ("class_weight='balanced'", "balanced")]:
    lr = LogisticRegression(C=0.05, class_weight=cw, max_iter=2000).fit(Xl, y_tr)
    p = lr.predict_proba(Xl_valid)[:, 1]
    demo[label] = {"valid_auc": roc_auc_score(y_va, p), "recall@0.5": recall_score(y_va, p >= 0.5),
                   "mean_predicted": p.mean(), "actual_default_rate": y_va.mean()}
pd.DataFrame(demo).T.round(3)

scale_pos_weight = 11.39


,valid_auc,recall@0.5,mean_predicted,actual_default_rate
no weighting,0.777,0.038,0.081,0.081
class_weight='balanced',0.777,0.706,0.404,0.081


**Reading the table:** weighting barely changes the AUC (ranking quality) but moves predicted probabilities far above
the real default rate. Weighting changes *where* the threshold falls, not how well the model ranks applicants.
That's why we calibrate the PDs and choose the threshold from business costs rather than relying on resampling.

In [9]:
write_json({"pos_weight": pos_weight, "n_numeric": len(num_sel), "n_categorical": len(cat_sel), "excluded": exclude,
            "split_sizes": {"train": len(train), "valid": len(valid), "test": len(test)}},
           paths["models"] / "preprocessing_info.json")
print("Saved feature contract (models/feature_lists.json) and split ids (data/processed/split_ids.parquet)")

Saved feature contract (models/feature_lists.json) and split ids (data/processed/split_ids.parquet)
